Uzbek-knowledgebased-RAG(Retrieval-Augmented Generation) system with a book 

In [ ]:
import os
import re
from pathlib import Path
from dotenv import load_dotenv
import fitz
import chromadb
from groq import Groq
from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL = "intfloat/multilingual-e5-large-instruct"
GROQ_MODEL      = "llama-3.3-70b-versatile"
CHUNK_SIZE      = 500
CHUNK_OVERLAP   = 50
TOP_K           = 3

load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

SYSTEM_PROMPT = (
    "Sen o'zbek tilidagi hujjatlar bo'yicha yordam beradigan assistantsan. "
    "Faqat berilgan hujjat bo'laklariga asoslanib javob ber. "
    "Agar javob hujjatlarda bo'lmasa, 'Bu ma'lumot hujjatda topilmadi' de. "
    "Har doim faqat o'zbek tilida javob ber."
)

print("done")

In [ ]:
print("Embedding modeli yuklanmoqda")

embedder    = SentenceTransformer(EMBEDDING_MODEL)
chroma      = chromadb.Client()
collection  = chroma.get_or_create_collection("uzbek_docs")
groq_client = Groq()

print("All models done")

In [ ]:
# took it from ai for making patterns more clear
kirill_lotin = {
    'а':'a', 'б':'b', 'в':'v', 'г':'g', 'д':'d',
    'е':'e', 'ё':'yo', 'ж':'j', 'з':'z', 'и':'i',
    'й':'y', 'к':'k', 'л':'l', 'м':'m', 'н':'n',
    'о':'o', 'п':'p', 'р':'r', 'с':'s', 'т':'t',
    'у':'u', 'ф':'f', 'х':'x', 'ц':'ts', 'ч':'ch',
    'ш':'sh', 'ъ':"'", 'э':'e', 'ю':'yu', 'я':'ya',
    'ў':'o\'', 'қ':'q', 'ғ':'g\'', 'ҳ':'h',
}

def kirill_to_lotin(text: str) -> str:
    result = ""
    for ch in text.lower():
        result += kirill_lotin.get(ch, ch)
    return result
def chunk_text(text: str, size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP) -> list[str]:
    text   = re.sub(r"\s+", " ", text).strip()
    chunks = []
    start  = 0
    while start < len(text):
        chunks.append(text[start : start + size])
        start += size - overlap
    return [c for c in chunks if len(c.strip()) > 20]


def load_txt(file_path: str) -> str:
    return Path(file_path).read_text(encoding="utf-8")


def load_pdf(file_path: str) -> str:
    doc  = fitz.open(file_path)
    text = ""
    for page in doc:
        text += page.get_text()
    doc.close()
    return text

def _save_to_db(chunks: list[str], name: str) -> int:
    embeddings = embedder.encode([f"passage: {c}" for c in chunks]).tolist()
    ids        = [f"{name}__{i}" for i in range(len(chunks))]

    collection.add(
        ids        = ids,
        embeddings = embeddings,
        documents  = chunks,
        metadatas  = [{"source": name, "chunk": i} for i in range(len(chunks))],
    )
    print(f"{name} yuklandi {len(chunks)} bo'lak")
    return len(chunks)

def add_document(file_path: str, doc_name: str = None) -> int:
    name   = doc_name or Path(file_path).stem
    text   = load_txt(file_path)
    chunks = chunk_text(text)
    return _save_to_db(chunks, name)

def add_pdf_kirill(file_path: str, doc_name: str = None) -> int:
    name   = doc_name or Path(file_path).stem
    text   = load_pdf(file_path)
    text   = kirill_to_lotin(text)
    chunks = chunk_text(text)
    return _save_to_db(chunks, name)


def search(query: str, top_k: int = TOP_K) -> list[dict]:
    vec = embedder.encode(f"query: {query}").tolist()
    res = collection.query(query_embeddings=[vec], n_results=top_k)
    return [
        {"text": doc, "source": meta["source"], "chunk": meta["chunk"]}
        for doc, meta in zip(res["documents"][0], res["metadatas"][0])
    ]


def ask(question: str) -> str:
    if collection.count() == 0:
        return "Hujjat yuklanmagan."

    chunks  = search(question)
    context = "\n\n".join(
        f"[{c['source']}, bo'lak {c['chunk']}]\n{c['text']}" for c in chunks
    )

    response = groq_client.chat.completions.create(
        model    = GROQ_MODEL,
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": f"Hujjat bo'laklari:\n{context}\n\nSavol: {question}"},
        ],
        max_tokens = 1024,
    )
    return response.choices[0].message.content
add_pdf_kirill("Alkimyogar.pdf")
print("All done sir")
javob = ask("Kitob bosh qahramoni nima bilan shug'ullanadi?")
print(javob)

In [ ]:
print("Savolingizni yozing (chiqish uchun 'quit' deb yozing):\n")

while True:
    savol = input("Savol: ").strip()
    if savol.lower() in ("chiq", "exit", "quit", ""):
        print("Xayr!")
        break
    print(f"\n-- {ask(savol)}\n")

In [ ]:
print(f"Jami saqlangan bo'laklar: {collection.count()}")